# D6 생성 모델 / Diffusion — 실습 (W13~14)

> ⚠️ **가장 먼저 — 화면 위 [Drive로 복사]를 누르세요.**
> 지금 보고 있는 것은 원본을 잠깐 띄운 **임시 사본**입니다. 복사하지 않으면 탭을 닫는 순간
> 채운 빈칸과 실행 결과가 **모두 사라집니다.** 복사본은 내 Google Drive에 저장되고,
> 원본은 바뀌지 않으니 마음껏 고쳐도 됩니다.

> 위에서부터 한 셀씩 `Shift+Enter`로 실행하세요. `___` 빈칸은 직접 채웁니다.
> (MNIST가 처음 실행 시 자동 다운로드됩니다. Colab에서 **런타임 → GPU**면 빠릅니다 — 없어도 동작, 학습에 수 분.)

**이 실습이 끝나면**
1. **AE**로 손글씨를 압축·복원(loss 0.0904→0.0265)하고, **노이즈 제거**(0.0473→0.0363)까지 확인한다
2. **VAE**로 무작위 z에서 **세상에 없던 숫자 16장**을 생성하고, **잠재 보간**(7→2 morph)으로 "정돈된 공간"을 확인한다
3. **한 픽셀의 확산 여행**(0.8 → 0.21 → −0.5)을 손계산으로 완주하고 순방향 노이징을 시각화한다

**7단계 멘탈모델 초점:** 모델 + 활용 (생성)

## Part A. 오토인코더(AE) — 압축했다 복원
784픽셀 → **잠재벡터 32차원**(24.5배 압축) → 784. 학습 목표는 **복원 ≈ 입력**(자기지도 — 이미지 자체가 문제집, 라벨 불필요).

In [ ]:
import warnings; warnings.filterwarnings('ignore')   # 출력 깔끔하게
import torch, torch.nn as nn                          # PyTorch
import matplotlib.pyplot as plt                       # 시각화
from torchvision import datasets, transforms          # MNIST
from torch.utils.data import DataLoader, Subset       # 배치 공급
torch.manual_seed(0)                                  # 재현성

tf = transforms.ToTensor()                            # [0,1] 텐서
train = Subset(datasets.MNIST('./data', train=True,  download=True, transform=tf), range(12000))  # 서브셋(속도)
test  = datasets.MNIST('./data', train=False, download=True, transform=tf)
tl = DataLoader(train, batch_size=128, shuffle=True)  # 배치 128
Xte = torch.stack([test[i][0] for i in range(16)]).view(16, -1)  # 시험 16장(평탄화)

class AE(nn.Module):                                  # 오토인코더(모래시계)
    def __init__(self):
        super().__init__()
        self.enc = nn.Sequential(nn.Linear(784,128), nn.ReLU(), nn.Linear(128,32))   # 784→32 압축
        self.dec = nn.Sequential(nn.Linear(32,128), nn.ReLU(), nn.Linear(128,784), nn.Sigmoid())  # 32→784 복원
    def forward(self, x):
        return self.dec(self.enc(x))                  # 압축→복원

ae = AE(); opt = torch.optim.Adam(ae.parameters(), 1e-3); crit = nn.MSELoss()
for ep in range(5):                                   # 5 epoch
    ae.train(); run = 0.0
    for x, _ in tl:                                   # D1c 5단계 루프 그대로
        x = x.view(x.size(0), -1); opt.zero_grad()
        loss = crit(ae(x), ___)                       # ✍️ 빈칸: 복원 목표 = 무엇 자신?
        loss.backward(); opt.step(); run += loss.item()
    print(f'AE epoch {ep+1}: loss={run/len(tl):.4f}')
print('AE 파라미터:', f'{sum(p.numel() for p in ae.parameters()):,}',
      '| 압축률 784→32 =', round(784/32, 1), '배')     # 209,968

In [ ]:
ae.eval()                                             # 평가 스위치(D1c)
with torch.no_grad():
    rec = ae(Xte)                                     # 시험 16장 재구성
fig, ax = plt.subplots(2, 8, figsize=(9, 2.4))        # 위: 원본 / 아래: 복원
for i in range(8):
    ax[0,i].imshow(Xte[i].view(28,28), cmap='gray'); ax[0,i].axis('off')
    ax[1,i].imshow(rec[i].view(28,28), cmap='gray'); ax[1,i].axis('off')
ax[0,0].set_title('original', loc='left', fontsize=9)
ax[1,0].set_title('reconstruction', loc='left', fontsize=9)
plt.tight_layout(); plt.show()

> loss 0.0904 → **0.0265** — 복원은 조금 흐릿하지만 숫자를 알아볼 수 있습니다. 24.5배 병목을 통과하려면 "그 숫자다움"(핵심 표현)만 남겨야 하기 때문 — 1학기 M8 PCA의 **비선형 판**입니다.

In [ ]:
torch.manual_seed(1)                                  # 재현성(노이즈용)
noisy = (Xte + 0.3*torch.randn_like(Xte)).clamp(0, 1) # 학습 때 본 적 없는 노이즈 낀 입력
with torch.no_grad():
    den = ae(___)                                     # ✍️ 빈칸: 무엇을 복원기에 넣나? (노이즈 낀 그 입력)
print('원본 대비 MSE — 노이즈 입력:', round(crit(noisy, Xte).item(), 4),
      '→ AE 출력:', round(crit(den, Xte).item(), 4))  # 0.0473 → 0.0363

fig, ax = plt.subplots(3, 8, figsize=(9, 3.6))        # 원본/노이즈/AE출력
for i in range(8):
    ax[0,i].imshow(Xte[i].view(28,28), cmap='gray'); ax[0,i].axis('off')
    ax[1,i].imshow(noisy[i].view(28,28), cmap='gray'); ax[1,i].axis('off')
    ax[2,i].imshow(den[i].view(28,28), cmap='gray'); ax[2,i].axis('off')
ax[0,0].set_title('original', loc='left', fontsize=9)
ax[1,0].set_title('noisy input', loc='left', fontsize=9)
ax[2,0].set_title('AE output', loc='left', fontsize=9)
plt.tight_layout(); plt.show()

> **배경 잡음이 걷혔습니다**(MSE 0.0473 → 0.0363) — 32차원 병목엔 노이즈가 들어갈 자리가 없습니다(활용: 노이즈 제거·이상 탐지).
> **그런데 생성은?** 무작위 z를 `ae.dec`에 넣으면 얼룩이 나오기 일쑤 — AE의 잠재공간은 훈련 데이터 자리 사이가 **빈 벌판**이라서. → 공간을 정돈하는 VAE로.

## Part B. VAE — 잠재공간을 분포로 정돈하면 생성이 된다
인코더가 **분포(μ, σ)** 를 출력하고, KL 벌점이 모든 분포를 **N(0, I)** 근방으로 정돈합니다.
재매개변수화 `z = μ + σ·ε` — **먼저 종이에서**: μ=[1.0, −0.5], σ=[0.5, 0.2], ε=[2, −1]이면 z = **[2.0, −0.7]**.

In [ ]:
torch.manual_seed(0)                                  # 재현성
class VAE(nn.Module):                                 # 변분 오토인코더
    def __init__(self, latent=20):
        super().__init__()
        self.fc1 = nn.Linear(784,128); self.mu = nn.Linear(128,latent); self.lv = nn.Linear(128,latent)
        self.fc2 = nn.Linear(latent,128); self.out = nn.Linear(128,784); self.latent = latent
    def encode(self, x):
        h = torch.relu(self.fc1(x)); return self.mu(h), self.lv(h)   # 평균, 로그분산
    def decode(self, z):
        return torch.sigmoid(self.out(torch.relu(self.fc2(z))))      # z→이미지
    def forward(self, x):
        mu, lv = self.encode(x); std = torch.exp(0.5*lv)
        z = mu + std * torch.___(std)                 # ✍️ 빈칸: 표준정규 노이즈 ε 주입(std와 같은 모양)
        return self.decode(z), mu, lv

def vae_loss(recon, x, mu, lv):                       # 복원오차 + KL(분포를 N(0,I)로 정돈)
    bce = nn.functional.binary_cross_entropy(recon, x, reduction='sum')
    kld = -0.5 * torch.sum(1 + lv - mu.pow(2) - lv.exp())
    return (bce + kld) / x.size(0)

vae = VAE(); opt = torch.optim.Adam(vae.parameters(), 1e-3)
for ep in range(10):                                  # 10 epoch
    vae.train(); run = 0.0
    for x, _ in tl:
        x = x.view(x.size(0), -1); opt.zero_grad()
        recon, mu, lv = vae(x); loss = vae_loss(recon, x, mu, lv)
        loss.backward(); opt.step(); run += loss.item()
    if ep in (0, 4, 9):
        print(f'VAE epoch {ep+1}: loss={run/len(tl):.1f}')
print('VAE 파라미터:', f'{sum(p.numel() for p in vae.parameters()):,}',
      '(AE 209,968과 거의 같은 예산 — 바뀐 건 규율)')  # 209,464

In [ ]:
vae.eval()                                            # 평가 스위치
torch.manual_seed(2)                                  # 재현성(샘플링용)
with torch.no_grad():
    z = torch.___(16, vae.latent)                     # ✍️ 빈칸: N(0,I)에서 뽑기 — KL이 정돈해 둔 그 분포
    gen = vae.decode(z)                               # 무작위 z → 새 숫자!
fig, ax = plt.subplots(2, 8, figsize=(9, 2.4))
for i in range(16):
    ax[i//8, i%8].imshow(gen[i].view(28,28).detach(), cmap='gray'); ax[i//8, i%8].axis('off')
fig.suptitle('VAE: digits generated from random latent z ~ N(0, I)', fontsize=10)
plt.tight_layout(); plt.show()

> 손실 **279.5 → 149.1 → 130.3**. 훈련 데이터에 없던 **새 손글씨 16장** — 다소 흐릿한 것이 VAE의 정직한 특성(복원오차가 "평균적으로 잘 그리기"를 선호). ε이 매번 다른 이미지를 만듭니다 — D5의 T>0 샘플링과 같은 정신.

In [ ]:
i1, i2 = 0, 1                                         # 시험 이미지 0번(7)과 1번(2)
with torch.no_grad():
    mu1, _ = vae.encode(Xte[i1:i1+1])                 # 7의 잠재 평균
    mu2, _ = vae.encode(Xte[i2:i2+1])                 # 2의 잠재 평균
    fig, ax = plt.subplots(1, 8, figsize=(10, 1.6))
    for k, a in enumerate(torch.linspace(0, 1, 8)):   # 두 점을 잇는 직선 위 8곳
        zk = (1-a)*mu1 + ___*mu2                      # ✍️ 빈칸: 보간 비율(a가 커질수록 2 쪽으로)
        img = vae.decode(zk)                          # 중간 지점을 디코딩
        ax[k].imshow(img.view(28,28), cmap='gray'); ax[k].axis('off')
        ax[k].set_title(f'a={a:.2f}', fontsize=8)
fig.suptitle('Latent interpolation: digit morphs smoothly', fontsize=10)
plt.tight_layout(); plt.show()

> **7이 매끄럽게 2로 변신** — 중간 어디를 찍어도 숫자처럼 생겼습니다. **빈 벌판이 사라졌다**(잠재공간이 연속으로 채워졌다)는 시각적 증거 — Part A의 AE 한계가 해소된 것.

## Part C. Diffusion — 순방향은 손계산이 된다 ⭐
노이즈 비율 t에서 **x_t = √(1−t)·x₀ + √t·ε** (정해진 규칙 — 학습 X).
**먼저 종이에서** 밝은 픽셀 x₀=0.8, 노이즈 ε=−0.5의 t=0.5를 완주해 보세요.

In [ ]:
x0_pix, eps = 0.8, -0.5                               # 밝은 픽셀 하나, 노이즈 하나
for t in [0.0, 0.5, 0.9, 1.0]:                        # 확산 여행 4정거장
    xt = ((1-t)**0.5)*x0_pix + (t**___)*eps           # ✍️ 빈칸: 노이즈 계수 = √t (t의 몇 제곱?)
    print(f't={t}: 신호 {round((1-t)**0.5, 2)} | 노이즈 {round(t**0.5, 2)} | 픽셀값 {round(xt, 4)}')

> **검산 포인트:** t=0.5 → 0.71×0.8 + 0.71×(−0.5) = **0.2121** (신호·노이즈 딱 절반, 0.71=√0.5), t=1 → **−0.5**(노이즈만). 곱셈으로 스러지는 신호 — D3a "신호의 세 운명"의 그 수학이 여기선 **일부러 지우는 도구**.

In [ ]:
torch.manual_seed(0)                                  # 재현성(노이즈 고정)
x0 = Xte[0].view(28, 28)                              # 깨끗한 원본(7)
noise = torch.randn_like(x0)                          # 고정 노이즈
steps = [0.0, 0.2, 0.4, 0.6, 0.8, 1.0]                # 노이즈 비율 t
fig, ax = plt.subplots(1, 6, figsize=(10, 2))
for i, t in enumerate(steps):
    xt = ((1-t)**0.5)*x0 + (t**0.5)*noise             # 픽셀 하나가 아니라 이미지 전체로
    ax[i].imshow(xt, cmap='gray'); ax[i].axis('off'); ax[i].set_title(f't={t:.1f}', fontsize=9)
fig.suptitle('Diffusion forward: clean image -> pure noise', fontsize=10)
plt.tight_layout(); plt.show()

### 역방향(생성)과 Stable Diffusion — 개념·데모
순방향이 **규칙**이므로 공짜 연습문제가 무한합니다: "노이즈 낀 x_t에서 **부은 ε을 맞혀 봐**"(정답은 우리가 부었으니 알고 있음 — 자기지도). 신경망이 **한 겹씩 걷어내기**를 배우면, 순수 노이즈에서 시작해 걷어내기를 반복 = **생성**. 텍스트 조건을 주면 "프롬프트 → 이미지"(Stable Diffusion) — **가져다 쓰기 계보의 4번째**(D2c ResNet → D4b DistilBERT → D5 LLM → D6 SD).

> 🖥️ **Colab에서 실행**(라이브러리·모델 다운로드가 큼, 수 GB — 이 노트북의 로컬 검증 범위는 여기까지):
> ```python
> !pip install diffusers transformers accelerate -q
> from diffusers import StableDiffusionPipeline
> import torch
> pipe = StableDiffusionPipeline.from_pretrained('stable-diffusion-v1-5/stable-diffusion-v1-5')
> pipe = pipe.to('cuda')                       # GPU 권장
> img = pipe('a cute corgi astronaut, digital art').images[0]
> img                                          # 생성된 이미지
> ```

### 🔹 GAN (개념)
![GAN](../../../_assets/gan.svg)

**생성자 G**(위조범)는 진짜 같은 가짜를 만들고, **판별자 D**(감별사)는 진짜/가짜를 가립니다. 둘의 군비 경쟁으로 함께 발전 — 선명하지만 **학습 불안정**(모드 붕괴). 2014~2020년경 왕좌 → 이후 주류는 Diffusion. (본 실습은 개념·도식까지)

## 🤖 AI 코파일럿 활용 (선택) — ai-native v1
막히면 AI 튜터에게 묻되, **먼저 스스로 생각**하고 답을 **실행으로 검증**하세요.

**좋은 질문 예시**
- "z = μ + σ·ε에서 [2.0, −0.7]을 내가 유도해 볼 테니 채점해 줘."
- "t=0.5에서 0.8이 0.2121이 되는 계산을 검산해 줘 (0.71=√0.5)."
- "AE가 생성에 약한 이유(빈 벌판)와 VAE의 해법(KL)을 내 말로 설명해 볼게."
- "보간 그림에서 중간 이미지가 '숫자처럼 생겼다'는 게 왜 중요한 증거인지 논박해 줘."

**가드레일**
1. 먼저 손으로 생각 → 그 다음 AI
2. AI 코드는 *왜 그런지* 설명할 수 있을 때만 사용
3. AI 출력은 실행으로 검증

## 정리 & 자가 점검

**오늘 한 일 3줄**
1. AE로 압축·복원(0.0904→0.0265)과 노이즈 제거(0.0473→0.0363)를 실측하고, 생성엔 약한 이유(빈 벌판)를 확인했다
2. VAE(같은 예산 209,464)로 무작위 z에서 **새 숫자 16장**을 만들고, 잠재 보간(7→2)으로 "정돈된 공간"을 봤다
3. 한 픽셀의 확산 여행(0.8→0.21→−0.5)을 손계산으로 완주하고 순방향 스트립을 그렸다

**스스로 점검**
- [ ] AE·VAE·Diffusion이 전부 자기지도인 이유(무엇이 정답인가)를 안다
- [ ] ε의 두 가지 역할(생성의 재료 + 역전파 통로)을 안다
- [ ] 보간 morph가 "빈 벌판 해소"의 증거인 이유를 안다
- [ ] Diffusion의 학습 대상이 역방향뿐임을 안다

**🔹심화 (선택)**
- **AE로 생성 실험:** `ae.dec(torch.randn(8, 32))`를 그려 보세요 — VAE 생성과 비교하면 "빈 벌판"이 보입니다.
- **latent 차원 실험:** VAE의 `latent`를 2로 줄이면 생성 품질이 어떻게 변할까요? (2차원이면 잠재공간 산점도도 그릴 수 있음)
- **보간 상대 바꾸기:** i1, i2를 다른 숫자 쌍으로 — 어떤 쌍이 가장 자연스럽게 변신하나요?
- **Colab:** Stable Diffusion 데모로 프롬프트→이미지를 직접. (다음 D7: 강화학습 — 2학기 완결편)